# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]
(https://colab.research.google.com/github/ErenSnowh/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns the validated model output into a **human-reviewed content action playbook**.
It includes ranked actions with reason codes, archetype→action mapping, intended use & limits,
human-review rules, cost/value thinking, monitoring/retrain triggers, and exports for the paper.

> **Claim standard:** All language in this playbook uses *observed*, *measured*, *directional*,
> and *decision-support* framing. No causal claims are made — this is cross-sectional data
> from one portfolio snapshot.

## 0. Setup

In [1]:
%pip install -q pandas numpy scikit-learn matplotlib

import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

# ── Paths ──
REPO_ROOT = Path(os.getcwd())
# Handle running from work/notebooks/ or repo root
if (REPO_ROOT / 'data' / 'raw').exists():
    pass  # already at repo root
elif (REPO_ROOT.parent.parent / 'data' / 'raw').exists():
    REPO_ROOT = REPO_ROOT.parent.parent
else:
    raise FileNotFoundError('Cannot find repo root — run from repo root or work/notebooks/')

RAW_CSV = REPO_ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
PIPELINE_QUEUE = REPO_ROOT / 'outputs' / 'refresh_queue.csv'
MODEL_RESULTS = REPO_ROOT / 'outputs' / 'model_results.json'
SUMMARY_JSON = REPO_ROOT / 'outputs' / 'summary.json'
W05_RESULTS = REPO_ROOT / 'work' / 'outputs' / 'w05_model_results.json'
W06_RESULTS = REPO_ROOT / 'work' / 'outputs' / 'w06_validation_audit_results.json'

WORK_OUT = REPO_ROOT / 'work' / 'outputs'
WORK_FIG = REPO_ROOT / 'work' / 'figures'
WORK_OUT.mkdir(parents=True, exist_ok=True)
WORK_FIG.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
print(f'Repo root: {REPO_ROOT}')
print(f'Pipeline queue exists: {PIPELINE_QUEUE.exists()}')



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\suzum\Downloads\flyrankinternproject\.venv\Scripts\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Repo root: C:\Users\suzum\Downloads\flyrankinternproject
Pipeline queue exists: True


## 0b. Load the pipeline queue and model results

In [2]:
# Load the final ranked queue from the reference pipeline
if PIPELINE_QUEUE.exists():
    queue = pd.read_csv(PIPELINE_QUEUE)
else:
    # Fall back: re-run pipeline first
    raise FileNotFoundError(
        'Run `python scripts/run_all.py` first to generate outputs/refresh_queue.csv'
    )

# Load model metrics receipts
with open(W05_RESULTS) as f:
    w05 = json.load(f)
with open(W06_RESULTS) as f:
    w06 = json.load(f)

print(f'Queue shape: {queue.shape}')
print(f'Columns: {list(queue.columns)}')
print(f'Actions: {queue["suggested_action"].value_counts().to_dict()}')
print(f'Confidence tiers: {queue["confidence"].value_counts().to_dict()}')


Queue shape: (30000, 28)
Columns: ['final_rank', 'content_id', 'client_id', 'final_refresh_score', 'best_model_name', 'best_model_probability', 'baseline_refresh_score', 'confidence', 'suggested_action', 'final_reason_codes', 'is_declining_label', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr', 'content_age_days', 'days_since_last_update', 'word_count', 'trend_direction', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']
Actions: {'monitor': 13069, 'refresh': 8207, 'refresh_and_review_ctr': 6655, 'refresh_and_review_engagement': 1987, 'expand_and_refresh': 82}
Confidence tiers: {'low': 15000, 'medium': 11424, 'high': 3576}


## 1. Ranked actions + reason codes

The queue ranks 30,000 content items by a blended score: 70% model probability
(decision tree, client-holdout validated) + 30% baseline rule score. Each item
carries transparent reason codes and a suggested action.

### Action definitions

| Action | Trigger | What to do |
|---|---|---|
| `refresh` | Model flags decline risk OR page is stale with demand | Rewrite/update the content, keep the URL |
| `refresh_and_review_ctr` | Above + low CTR on a visible page (position ≤ 20, CTR < 0.5%) | Refresh AND rewrite the title/meta to improve click-through |
| `refresh_and_review_engagement` | Above + low engagement/scroll on a page with 30+ sessions | Refresh AND audit on-page UX (layout, readability, CTA placement) |
| `expand_and_refresh` | Thin page (< 1,200 words) with 250+ impressions | Substantially expand content depth, then refresh |
| `monitor` | No urgent signals; page is performing adequately | No immediate action — re-score in the next cycle |

### Reason code dictionary

| Reason code | Meaning |
|---|---|
| `declining_with_demand` | Trend is down but impressions ≥ 100 — this page still has search demand |
| `stale_visible_page` | Not updated in 180+ days with 500+ impressions |
| `thin_visible_page` | Under 1,200 words with 250+ impressions |
| `page_one_decay_risk` | Page-1 position (≤ 10) but content is 180+ days old |
| `low_ctr_visible_page` | CTR < 0.5% with position ≤ 20 and 500+ impressions |
| `low_engagement_visible_page` | Engagement or scroll rate < 30% with 30+ sessions |
| `model_decline_risk` | Model probability ≥ 0.65 (above the positive-class threshold) |
| `visible_model_opportunity` | Model probability ≥ 0.5 AND 500+ impressions |
| `ctr_review_candidate` | Same as `low_ctr_visible_page` — added in final merge |
| `engagement_review_candidate` | Same as `low_engagement_visible_page` — added in final merge |
| `general_refresh_review` | No specific signal — included for completeness |

### Archetype → action mapping

Pages cluster into observable archetypes based on their reason-code combinations.
Below are the five most common patterns and the recommended workflow for each.

In [3]:
# ── 1a. Archetype analysis ──
# Count reason-code combinations to find natural archetypes
archetype_counts = (
    queue
    .assign(archetype=queue['final_reason_codes'])
    .groupby('archetype')
    .agg(
        count=('content_id', 'count'),
        mean_score=('final_refresh_score', 'mean'),
        pct_declining=('is_declining_label', 'mean'),
        median_impressions=('impressions_90d', 'median'),
    )
    .sort_values('count', ascending=False)
)

print('=== Top 10 page archetypes (by reason-code combination) ===')
print(archetype_counts.head(10).to_string())
print(f'\nTotal distinct archetypes: {len(archetype_counts)}')


=== Top 10 page archetypes (by reason-code combination) ===
                                                                                                               count  mean_score  pct_declining  median_impressions
archetype                                                                                                                                                          
general_refresh_review                                                                                          7701   32.062558       0.239449                29.0
page_one_decay_risk                                                                                             2364   33.203356       0.332064                11.0
declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate    1962   67.491090       1.000000              1548.5
declining_with_demand|model_decline_risk                                                                        1560   6

In [4]:
# ── 1b. Action mix summary ──
action_summary = (
    queue
    .groupby('suggested_action')
    .agg(
        count=('content_id', 'count'),
        mean_score=('final_refresh_score', 'mean'),
        pct_declining=('is_declining_label', 'mean'),
        median_impressions=('impressions_90d', 'median'),
        pct_high_conf=('confidence', lambda x: (x == 'high').mean()),
    )
    .sort_values('count', ascending=False)
)

print('=== Action mix ===')
print(action_summary.to_string())


=== Action mix ===
                               count  mean_score  pct_declining  median_impressions  pct_high_conf
suggested_action                                                                                  
monitor                        13069   36.246636       0.202081                55.0       0.000000
refresh                         8207   57.781087       0.684781               538.0       0.093335
refresh_and_review_ctr          6655   62.699032       0.917355              2477.0       0.277385
refresh_and_review_engagement   1987   62.739267       0.932058              6589.0       0.481127
expand_and_refresh                82   52.188320       0.536585               708.5       0.097561


In [5]:
# ── 1c. Top-20 queue preview (the first pages a reviewer would see) ──
preview_cols = [
    'final_rank', 'final_refresh_score', 'best_model_probability',
    'suggested_action', 'final_reason_codes', 'confidence',
    'impressions_90d', 'sessions_90d', 'avg_position', 'ctr',
    'content_age_days', 'days_since_last_update',
]
top20 = queue.head(20)[preview_cols]
print('=== Top 20 refresh queue ===')
print(top20.to_string(index=False))


=== Top 20 refresh queue ===
 final_rank  final_refresh_score  best_model_probability              suggested_action                                                                                                                                                   final_reason_codes confidence  impressions_90d  sessions_90d  avg_position  ctr  content_age_days  days_since_last_update
          1            81.928467                0.786247        refresh_and_review_ctr declining_with_demand|low_ctr_visible_page|low_engagement_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate|engagement_review_candidate       high            12834            66           6.8 0.05               165                     104
          2            81.728449                0.792117        refresh_and_review_ctr                                                         declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate       hig

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── 1d. Action mix chart ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Action distribution
action_cts = queue['suggested_action'].value_counts()
colors_action = ['#426B69', '#6F4E7C', '#4E79A7', '#B07AA1', '#8C6BB1']
axes[0].barh(action_cts.index[::-1], action_cts.values[::-1],
             color=colors_action[:len(action_cts)])
axes[0].set_xlabel('Number of pages')
axes[0].set_title('Action distribution')
for i, v in enumerate(action_cts.values[::-1]):
    axes[0].text(v + 100, i, f'{v:,}', va='center', fontsize=9)

# Confidence distribution
conf_cts = queue['confidence'].value_counts().reindex(['high', 'medium', 'low'], fill_value=0)
colors_conf = ['#2d6a4f', '#74c69d', '#d4d4d4']
axes[1].barh(conf_cts.index[::-1], conf_cts.values[::-1], color=colors_conf[::-1])
axes[1].set_xlabel('Number of pages')
axes[1].set_title('Confidence tier distribution')
for i, v in enumerate(conf_cts.values[::-1]):
    axes[1].text(v + 100, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
fig.savefig(WORK_FIG / 'w07_action_confidence_mix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {WORK_FIG / "w07_action_confidence_mix.png"}')


Saved: C:\Users\suzum\Downloads\flyrankinternproject\work\figures\w07_action_confidence_mix.png


C:\Users\suzum\AppData\Local\Temp\ipykernel_30080\2412993164.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ── 1e. Reason code frequency ──
reason_counts = {}
for codes in queue['final_reason_codes']:
    for r in str(codes).split('|'):
        reason_counts[r] = reason_counts.get(r, 0) + 1

reason_series = pd.Series(reason_counts).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(reason_series.index, reason_series.values, color='#4E79A7')
ax.set_xlabel('Pages with this reason code')
ax.set_title('Reason code frequency across the full queue')
for i, v in enumerate(reason_series.values):
    ax.text(v + 100, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
fig.savefig(WORK_FIG / 'w07_reason_code_frequency.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {WORK_FIG / "w07_reason_code_frequency.png"}')


Saved: C:\Users\suzum\Downloads\flyrankinternproject\work\figures\w07_reason_code_frequency.png


C:\Users\suzum\AppData\Local\Temp\ipykernel_30080\3781286070.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Intended use and limits

### Who uses this playbook

- **Content editors / SEO strategists** at FlyRank client teams — they pick which pages
  to refresh in a given sprint.
- **Account managers** reviewing client portfolios — the queue surfaces the pages most
  likely to benefit from editorial attention.

### How to use it

1. Sort by `final_rank` (already sorted).
2. Start from the top — high-confidence, high-score pages first.
3. Read the `suggested_action` and `final_reason_codes` to understand *why* the page was flagged.
4. **Open the actual page** before acting — the model scores behavioral signals, not content quality.
5. Apply the action if the editorial context confirms the signal.

### Intended scope

- **Population:** The 30,000-item anonymized starter dataset (32 pseudonymized clients,
  trailing 90-day metrics). This is one snapshot — it captures the state of these pages
  at export time, not their future trajectory.
- **Task:** Binary classification — "is this page's impression trend declining?" —
  used as a *proxy* for "would editorial attention on this page be worthwhile?"
- **Validation:** Client-holdout split (∼20% of clients held out), so the model has
  been tested on clients it never trained on.

### Known limits

| Limit | Why it matters |
|---|---|
| **Cross-sectional, not causal** | We observed that pages with certain signal patterns are associated with declining trends. We did NOT test whether refreshing those pages reverses the decline. An A/B test would be needed to establish causation. |
| **One snapshot in time** | Seasonal effects, algorithm updates, or market shifts may change which signals matter. The queue is a point-in-time recommendation. |
| **32 clients only** | Generalization to clients with very different content profiles (e.g., e-commerce vs. B2B SaaS) has not been tested. |
| **No content-quality features** | The model uses behavioral/structural signals (impressions, position, age, CTR). It cannot assess whether the content is factually accurate, well-written, or topically relevant. |
| **Label is a proxy** | `is_declining_label` is defined as impression trend direction = "down". Impression decline doesn't always mean the content needs refreshing (e.g., seasonality, sunsetting a product). |
| **Precision@50 on client-holdout ≈ 0.68** | 32 of the top 50 predictions are correct — useful, but 18 are false positives that a human reviewer filters out. |
| **Rate columns are ×100** | `ctr = 0.76` means 0.76%, not 76%. Misreading this leads to wrong CTR-based actions. |

In [8]:
# ── 2. Quantify the limits ──
base_rate = w05['base_rate']
best_p50 = w05['results']['decision_tree']['Precision@50']

print('=== Quantified limits ===')
print(f'Base rate (random precision): {base_rate:.1%}')
print(f'Model Precision@50 (client-holdout): {best_p50:.1%}')
print(f'Lift over random: {best_p50 / base_rate:.1f}×')
print(f'False-positive rate in top 50: {1 - best_p50:.1%}')
print(f'→ Of every 50 pages the model flags as top-priority,')
print(f'  ~{int(50 * best_p50)} are genuinely declining and ~{int(50 * (1 - best_p50))} are not.')
print(f'  A human reviewer eliminates the {int(50 * (1 - best_p50))} false positives.')
print()
print(f'Total pages: {len(queue):,}')
print(f'High-confidence actions: {(queue["confidence"] == "high").sum():,}')
print(f'Pages flagged for active action (not monitor): '
      f'{(queue["suggested_action"] != "monitor").sum():,}')


=== Quantified limits ===
Base rate (random precision): 39.1%
Model Precision@50 (client-holdout): 68.0%
Lift over random: 1.7×
False-positive rate in top 50: 32.0%
→ Of every 50 pages the model flags as top-priority,
  ~34 are genuinely declining and ~15 are not.
  A human reviewer eliminates the 15 false positives.

Total pages: 30,000
High-confidence actions: 3,576
Pages flagged for active action (not monitor): 16,931


## 3. Human review + the no-go list

### What a human MUST check before acting

The model ranks pages by statistical signals. A human reviewer must verify:

1. **Content relevance** — Is the topic still relevant to the client's business?
   A page about a deprecated product should be sunset, not refreshed.
2. **Seasonal context** — Is the decline seasonal? (e.g., "tax filing tips" declining
   in July is normal.) Check if the keyword has seasonal volume patterns.
3. **Recent updates** — Has the page been updated since the data export? The model
   may flag a page that was already fixed.
4. **Cannibalization** — Is another page on the same site targeting the same query?
   Refreshing the wrong page can make cannibalization worse.
5. **Client priorities** — The client may have strategic reasons to deprioritize
   certain content (e.g., a service they're sunsetting).
6. **Cost/effort ratio** — A thin page (< 1,200 words) flagged as `expand_and_refresh`
   may require 4–8 hours of writing. Is the keyword volume worth that investment?

### The no-go list — what should NEVER be automated

| ❌ Never automate | Why |
|---|---|
| Publishing a refresh without human review | The model doesn't read the content — it scores signals |
| Deleting or redirecting pages based on model scores | Deletion is irreversible; the model has a 32% false-positive rate in top-50 |
| Setting budgets or SLAs from model scores | The score is ordinal (ranking), not a dollar value |
| Communicating to clients that their content "will improve" if refreshed | Cross-sectional data; no causal evidence of refresh → improvement |
| Applying the queue across new clients without re-validation | 32 training clients ≠ the full population |
| Treating `monitor` as "this page is healthy" | Monitor means no urgent signal was detected — not that the page is performing well |

In [9]:
# ── 3. Cost/value sizing ──
# Estimate editorial effort per action tier
effort_hours = {
    'expand_and_refresh': 6.0,   # Substantial rewrite
    'refresh_and_review_ctr': 3.0,  # Content + meta rewrite
    'refresh_and_review_engagement': 3.5,  # Content + UX audit
    'refresh': 2.0,               # Standard content update
    'monitor': 0.0,               # No action this cycle
}

action_effort = (
    queue
    .groupby('suggested_action')
    .agg(count=('content_id', 'count'))
    .assign(
        est_hours_each=lambda d: d.index.map(effort_hours),
        total_hours=lambda d: d['count'] * d['est_hours_each'],
    )
)

print('=== Cost/value sizing (full queue) ===')
print(action_effort.to_string())
print(f'\nTotal estimated hours if ALL flagged pages were refreshed: '
      f'{action_effort["total_hours"].sum():,.0f}')
print()

# Realistic: only high-confidence pages
high_conf = queue[queue['confidence'] == 'high']
high_effort = (
    high_conf
    .groupby('suggested_action')
    .agg(count=('content_id', 'count'))
    .assign(
        est_hours_each=lambda d: d.index.map(effort_hours),
        total_hours=lambda d: d['count'] * d['est_hours_each'],
    )
)
print('=== Cost/value sizing (high-confidence only) ===')
print(high_effort.to_string())
print(f'\nTotal estimated hours for high-confidence pages: '
      f'{high_effort["total_hours"].sum():,.0f}')
print(f'This is the realistic starting batch for a team sprint.')


=== Cost/value sizing (full queue) ===
                               count  est_hours_each  total_hours
suggested_action                                                 
expand_and_refresh                82             6.0        492.0
monitor                        13069             0.0          0.0
refresh                         8207             2.0      16414.0
refresh_and_review_ctr          6655             3.0      19965.0
refresh_and_review_engagement   1987             3.5       6954.5

Total estimated hours if ALL flagged pages were refreshed: 43,826

=== Cost/value sizing (high-confidence only) ===
                               count  est_hours_each  total_hours
suggested_action                                                 
expand_and_refresh                 8             6.0         48.0
refresh                          766             2.0       1532.0
refresh_and_review_ctr          1846             3.0       5538.0
refresh_and_review_engagement    956             3.5

## 4. Monitoring / retrain triggers

### When the recommendations go stale

The queue is a point-in-time snapshot. It should be re-generated:

| Trigger | Check | Threshold |
|---|---|---|
| **Calendar staleness** | Time since last data export | > 90 days |
| **Distribution shift** | Compare current feature distributions vs. training data | Kolmogorov-Smirnov p < 0.01 on any top-5 feature |
| **Label drift** | Observed declining rate in new data vs. training base rate (54.2%) | Shift > 5 percentage points |
| **Precision decay** | Re-evaluate Precision@50 on a new holdout sample | Drops below 0.50 (below 1.3× the base rate) |
| **Client portfolio change** | New clients onboarded or existing clients churned | Any change in client mix |
| **Major algorithm update** | Google announces a core search update | Within 30 days of the update |

### Lightweight monitoring protocol

1. **Monthly:** Compare the action distribution of the new export vs. this baseline.
   If the `monitor` bucket grows by > 10 pp (indicating fewer actionable pages),
   the model may be losing signal.
2. **Quarterly:** Re-run the model on the latest 90-day window and compare
   Precision@20 and Precision@50 against the benchmarks below.
3. **On retrain:** Hold out the same ~20% client fraction and compare metrics.
   A retrained model that doesn't beat the current one gets discarded.

In [10]:
# ── 4a. Feature distribution baselines for drift detection ──
top_features = ['days_with_impressions', 'content_age_days', 'days_with_sessions',
                'avg_position', 'scroll_rate']

drift_baselines = {}
for feat in top_features:
    if feat in queue.columns:
        vals = queue[feat].dropna()
        drift_baselines[feat] = {
            'mean': round(float(vals.mean()), 4),
            'std': round(float(vals.std()), 4),
            'p25': round(float(vals.quantile(0.25)), 4),
            'median': round(float(vals.median()), 4),
            'p75': round(float(vals.quantile(0.75)), 4),
        }

print('=== Feature distribution baselines (for future drift comparison) ===')
drift_df = pd.DataFrame(drift_baselines).T
print(drift_df.to_string())


=== Feature distribution baselines (for future drift comparison) ===
                      mean       std    p25  median    p75
content_age_days  256.1678  132.7079  132.0   236.0  333.0
avg_position       16.3424   15.2168    6.2    10.8   22.3


In [11]:
# ── 4b. Action-mix baseline (for distribution shift detection) ──
action_baseline = queue['suggested_action'].value_counts(normalize=True).round(4)
conf_baseline = queue['confidence'].value_counts(normalize=True).round(4)

print('=== Action mix baseline (proportions) ===')
print(action_baseline.to_string())
print()
print('=== Confidence tier baseline (proportions) ===')
print(conf_baseline.to_string())
print()
print('These proportions serve as the reference. If a future re-run shows monitor')
print('growing by >10pp or high-confidence shrinking by >5pp, investigate.')


=== Action mix baseline (proportions) ===
suggested_action
monitor                          0.4356
refresh                          0.2736
refresh_and_review_ctr           0.2218
refresh_and_review_engagement    0.0662
expand_and_refresh               0.0027

=== Confidence tier baseline (proportions) ===
confidence
low       0.5000
medium    0.3808
high      0.1192

These proportions serve as the reference. If a future re-run shows monitor
growing by >10pp or high-confidence shrinking by >5pp, investigate.


In [12]:
# ── 4c. Performance benchmarks to compare against ──
print('=== Performance benchmarks (retrain must beat these) ===')
print(f'Decision Tree (client-holdout):')
dt = w05['results']['decision_tree']
print(f'  Precision@20: {dt["Precision@20"]}')
print(f'  Precision@50: {dt["Precision@50"]}')
print(f'  ROC-AUC:      {dt["ROC-AUC"]}')
print()
print(f'Random Forest (client-holdout):')
rf = w05['results']['random_forest']
print(f'  Precision@20: {rf["Precision@20"]}')
print(f'  Precision@50: {rf["Precision@50"]}')
print(f'  ROC-AUC:      {rf["ROC-AUC"]}')
print()
print(f'Baseline rules:')
bl = w05['results']['baseline_rules']
print(f'  Precision@20: {bl["Precision@20"]}')
print(f'  Precision@50: {bl["Precision@50"]}')
print()
print(f'Base rate (random): {w05["base_rate"]:.1%}')
print(f'\nA retrained model must exceed baseline rules on Precision@50')
print(f'and should aim for ≥ the current decision tree benchmarks.')


=== Performance benchmarks (retrain must beat these) ===
Decision Tree (client-holdout):
  Precision@20: 0.8
  Precision@50: 0.68
  ROC-AUC:      0.7415

Random Forest (client-holdout):
  Precision@20: 0.7
  Precision@50: 0.68
  ROC-AUC:      0.7474

Baseline rules:
  Precision@20: 0.15
  Precision@50: 0.24

Base rate (random): 39.1%

A retrained model must exceed baseline rules on Precision@50
and should aim for ≥ the current decision tree benchmarks.


## 5. Exports for the paper

This section writes the final queue and summary files to `work/outputs/`.
These are the exact files the research paper will reference.

In [13]:
# ── 5a. Export the playbook queue to work/outputs/ ──
playbook_cols = [
    'final_rank', 'content_id', 'client_id',
    'final_refresh_score', 'best_model_probability',
    'confidence', 'suggested_action', 'final_reason_codes',
    'is_declining_label',
    'impressions_90d', 'clicks_90d', 'sessions_90d',
    'avg_position', 'ctr',
    'content_age_days', 'days_since_last_update',
    'word_count', 'trend_direction',
    'content_type', 'main_intent',
]
playbook_queue = queue[[c for c in playbook_cols if c in queue.columns]]
playbook_path = WORK_OUT / 'w07_playbook_queue.csv'
playbook_queue.to_csv(playbook_path, index=False)
print(f'Wrote playbook queue: {playbook_path} ({len(playbook_queue):,} rows)')


Wrote playbook queue: C:\Users\suzum\Downloads\flyrankinternproject\work\outputs\w07_playbook_queue.csv (30,000 rows)

In [14]:
# ── 5b. Export the playbook receipt JSON ──
receipt = {
    'notebook': 'w07_action_playbook.ipynb',
    'assignment': 'ML-10 Content Action Playbook',
    'random_state': RANDOM_STATE,
    'total_pages_scored': int(len(queue)),
    'action_distribution': queue['suggested_action'].value_counts().to_dict(),
    'confidence_distribution': queue['confidence'].value_counts().to_dict(),
    'high_confidence_pages': int((queue['confidence'] == 'high').sum()),
    'pages_needing_action': int((queue['suggested_action'] != 'monitor').sum()),
    'model_used': w05['best_model'],
    'model_precision_at_50': w05['results'][w05['best_model']]['Precision@50'],
    'base_rate': w05['base_rate'],
    'validation_strategy': w05['split_strategy'],
    'leakage_assertion_passed': w06['leakage_assertion_passed'],
    'claim_language_standard': 'observed, measured, directional, decision-support',
    'no_go_items': [
        'Auto-publish without human review',
        'Delete or redirect pages from model scores alone',
        'Set budgets or SLAs from model scores',
        'Promise clients that refresh will improve rankings',
        'Apply queue to new clients without re-validation',
    ],
    'retrain_triggers': [
        'Data > 90 days old',
        'Feature distribution shift (KS p < 0.01)',
        'Label drift > 5pp from base rate',
        'Precision@50 drops below 0.50',
        'Client portfolio change',
        'Major search algorithm update',
    ],
    'drift_baselines': drift_baselines,
    'action_mix_baseline': queue['suggested_action'].value_counts(normalize=True).round(4).to_dict(),
    'exports': [
        'work/outputs/w07_playbook_queue.csv',
        'work/outputs/w07_playbook_receipt.json',
        'work/figures/w07_action_confidence_mix.png',
        'work/figures/w07_reason_code_frequency.png',
    ],
}

receipt_path = WORK_OUT / 'w07_playbook_receipt.json'
with open(receipt_path, 'w') as f:
    json.dump(receipt, f, indent=2)
print(f'Wrote receipt: {receipt_path}')


Wrote receipt: C:\Users\suzum\Downloads\flyrankinternproject\work\outputs\w07_playbook_receipt.json


In [15]:
# ── 5c. Skeptic's audit: top-5 picks — what would make them wrong? ──
print('=== Skeptic\'s audit: top 5 picks ===')
for i, row in queue.head(5).iterrows():
    print(f'\nRank {row["final_rank"]}: score={row["final_refresh_score"]:.1f}, '
          f'action={row["suggested_action"]}')
    print(f'  Reasons: {row["final_reason_codes"]}')
    print(f'  Impressions={int(row["impressions_90d"]):,}, '
          f'Sessions={int(row["sessions_90d"])}, '
          f'Position={row["avg_position"]:.1f}, '
          f'CTR={row["ctr"]:.2f}%')
    print(f'  Age={int(row["content_age_days"])}d, '
          f'Last update={int(row["days_since_last_update"])}d ago')
    # What would make this pick wrong?
    critiques = []
    if row['impressions_90d'] < 1000:
        critiques.append('Moderate volume — decline could be noise')
    if row['days_since_last_update'] < 60:
        critiques.append('Recently updated — decline may be post-update settling')
    if row['avg_position'] > 20:
        critiques.append('Deep position — limited upside from content refresh alone')
    if row['ctr'] > 2.0:
        critiques.append('CTR is already reasonable — title/meta may not be the issue')
    if not critiques:
        critiques.append('No obvious red flags — this pick appears well-supported')
    print(f'  ⚠️ What would make it wrong: {"; ".join(critiques)}')


=== Skeptic's audit: top 5 picks ===

Rank 1: score=81.9, action=refresh_and_review_ctr
  Reasons: declining_with_demand|low_ctr_visible_page|low_engagement_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate|engagement_review_candidate
  Impressions=12,834, Sessions=66, Position=6.8, CTR=0.05%
  Age=165d, Last update=104d ago
  ⚠️ What would make it wrong: No obvious red flags — this pick appears well-supported

Rank 2: score=81.7, action=refresh_and_review_ctr
  Reasons: declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate
  Impressions=8,064, Sessions=23, Position=3.8, CTR=0.07%
  Age=139d, Last update=104d ago
  ⚠️ What would make it wrong: No obvious red flags — this pick appears well-supported

Rank 3: score=81.6, action=refresh_and_review_ctr
  Reasons: declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate
  Impressions=2,498, Sessions=9, Posi

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Exports written to `work/outputs/` (queue CSV + receipt JSON + figures)
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.